# WM-811K 비라벨 자기지도 사전학습

목표: 버려졌던 비라벨 wafer map을 이용해 CNN encoder를 contrastive 방식으로 사전학습한 뒤, 기존 분류 기준선과 **validation-only**로 비교합니다.

누수 방지 계약:
- 비라벨 샘플은 frozen split의 **train lot**에 속한 것만 사용합니다.
- validation/test lot과 lot을 알 수 없는 비라벨 샘플은 제외합니다.
- fine-tuning은 labeled train만, 선택은 validation만 사용합니다.
- 기존 test는 이 실험에서 다시 열지 않습니다. 통과해도 외부 holdout 검증 전에는 배포 모델을 바꾸지 않습니다.

예상 시간: T4에서 추출 5~15분, 사전학습과 3시드 fine-tuning 25~45분.

In [ ]:
!pip install -q -U kagglehub 'pandas==2.2.3' 'scikit-learn==1.6.1' Pillow

from google.colab import drive, files, userdata
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 T4 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## 1. 실행 번들 1개 업로드

로컬 프로젝트의 노트북/wm811k_ssl_colab_bundle.zip 하나만 선택합니다. 파일명 뒤에 (1)이 붙어도 처리합니다.

In [ ]:
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(zip_names) == 1, f'ZIP 하나를 선택하세요: {list(uploaded)}'
bundle = Path('/content/wm811k_ssl_bundle.zip')
bundle.write_bytes(uploaded[zip_names[0]])
bundle_dir = Path('/content/wm811k_ssl_bundle')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
with zipfile.ZipFile(bundle) as archive:
    assert all('..' not in Path(name).parts for name in archive.namelist())
    archive.extractall(bundle_dir)
required = ['preprocess_wm811k.py', 'extract_wm811k_train_lot_unlabeled.py', 'train_wm811k_cnn.py', 'train_wm811k_ssl_candidate.py', 'split_assignments.csv', 'baseline_reproducibility_summary.json', 'baseline_class_recall_summary.json']
for name in required:
    assert (bundle_dir / name).is_file(), f'번들 누락: {name}'
print('번들 확인 완료:', required)

## 2. 원본 LSWMD.pkl 받기

Colab Secrets에 KAGGLE_API_TOKEN을 두고 노트북 접근을 허용합니다. 토큰 문자열은 출력하거나 공유하지 않습니다.

In [ ]:
import kagglehub
token = userdata.get('KAGGLE_API_TOKEN')
assert token, 'Colab Secrets에 KAGGLE_API_TOKEN을 추가하세요.'
os.environ['KAGGLE_API_TOKEN'] = token
download_root = Path('/content/wm811k_kaggle')
download_root.mkdir(parents=True, exist_ok=True)
downloaded = Path(kagglehub.dataset_download('qingyi/wm811k-wafer-map', path='LSWMD.pkl', output_dir=str(download_root)))
candidates = ([downloaded] if downloaded.is_file() else []) + list(download_root.rglob('LSWMD.pkl'))
assert candidates, f'LSWMD.pkl을 찾지 못했습니다: {downloaded}'
PICKLE = candidates[0]
print('원본:', PICKLE, f'{PICKLE.stat().st_size / 1024**2:.1f} MB')

## 3. train lot 비라벨 샘플 추출

최대 100,000개를 64×64 uint8로 변환합니다. 출력은 Drive에 저장되므로 런타임이 끊겨도 재사용할 수 있습니다.

In [ ]:
PROCESSED = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
UNLABELED = Path('/content/drive/MyDrive/AI_project/WM811K_ssl_unlabeled')
for name in ['wafer_maps_64.npy', 'labels.npy']:
    assert (PROCESSED / name).is_file(), f'1단계 전처리 출력이 없습니다: {PROCESSED / name}'
unlabeled_maps = UNLABELED / 'unlabeled_train_lot_maps_64.npy'
unlabeled_summary = UNLABELED / 'unlabeled_summary.json'
if unlabeled_maps.is_file() and unlabeled_summary.is_file():
    print('기존 비라벨 추출 결과 재사용:', UNLABELED)
else:
    command = [sys.executable, str(bundle_dir / 'extract_wm811k_train_lot_unlabeled.py'), '--input', str(PICKLE), '--assignments', str(bundle_dir / 'split_assignments.csv'), '--output-dir', str(UNLABELED), '--maximum-samples', '100000', '--seed', '42']
    subprocess.run(command, check=True, cwd=bundle_dir)
receipt = json.loads(unlabeled_summary.read_text(encoding='utf-8'))
assert receipt['validation_or_test_lots_used'] is False
assert receipt['labels_used'] is False
print(json.dumps(receipt, ensure_ascii=False, indent=2))

## 4. contrastive 사전학습 + validation-only 3시드 비교

기준선은 기존 ce_sqrt_balanced 3시드 평균입니다. 아래 코드는 test loader를 만들지 않습니다.

In [ ]:
OUTPUT = Path('/content/drive/MyDrive/AI_project/WM811K_ssl_experiment/contrastive_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(bundle_dir / 'train_wm811k_ssl_candidate.py'), '--maps', str(PROCESSED / 'wafer_maps_64.npy'), '--labels', str(PROCESSED / 'labels.npy'), '--assignments', str(bundle_dir / 'split_assignments.csv'), '--unlabeled-maps', str(unlabeled_maps), '--unlabeled-summary', str(unlabeled_summary), '--baseline-summary', str(bundle_dir / 'baseline_reproducibility_summary.json'), '--baseline-class-recall', str(bundle_dir / 'baseline_class_recall_summary.json'), '--output-dir', str(OUTPUT), '--pretrain-epochs', '8', '--pretrain-batch-size', '512', '--epochs', '20', '--batch-size', '256', '--num-workers', '2', '--finetune-seeds', '17', '42', '2026']
subprocess.run(command, check=True, cwd=bundle_dir)
comparison = json.loads((OUTPUT / 'ssl_candidate_comparison.json').read_text(encoding='utf-8'))
print('결론:', comparison['decision'])
print('모든 validation guardrail 통과:', comparison['all_guardrails_passed'])
print(json.dumps(comparison['candidate_validation_metrics'], ensure_ascii=False, indent=2))

## 5. 전달용 결과 ZIP 다운로드

대용량 비라벨 배열은 제외하고 학습 영수증·곡선·validation 예측·checkpoint를 압축합니다. 다운로드한 ZIP을 로컬 프로젝트 결과물/wm811k/colab_가져오기/에 압축을 풀지 말고 넣으세요.

In [ ]:
zip_base = Path('/content/wm811k_ssl_validation_results')
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT))
print('ZIP:', zip_path, f'{zip_path.stat().st_size / 1024**2:.1f} MB')
files.download(str(zip_path))